# Test Kümesi Değerlendirmesi (yalnız test)

Eğitilmiş `best.pt` ile **yalnızca** dondurulmuş bölmenin test kümesi değerlendirilir ve `metrics_test.csv` üretilir:

| metrik | açıklama | kaynak |
|---|---|---|
| `mAP50`, `mAP50-95` (all) | tüm nesneler | Ultralytics `val(split="test")` |
| `mAP50` (sınıf bazında) | her sınıf için | Ultralytics |
| `AP_small_50-95`, `AP_small_50` | alanı ≤ 32×32 px olan nesneler (COCO alan filtresi, orijinal piksel) | pycocotools |

Ayrıca: sınıf bazında küçük nesne AP'si, GT sayıları, COCO çapraz kontrolü, `coco_gt_test.json`, `predictions_test.json`, **20 örnek çıktı** (`ornek_ciktilar/`). Hepsi Drive'da `uav_proje/test_results/` altına da kopyalanır.

Güvenceler: manifest parmak izi kontrolü, test görsellerinin MD5 kontrolü, ağırlığın **aynı bölmeyle** eğitildiği kontrolü (`runs/.../split_info.json`).

In [ ]:
%pip install -q ultralytics pycocotools

In [ ]:
# ==== TEST KÜMESİ DEĞERLENDİRMESİ — AYARLAR ====
# Yalnız dondurulmuş bölmenin TEST kümesi kullanılır. Ağırlık başka bir bölmeyle eğitildiyse durur (sızıntı koruması).
import os, sys, csv, json, hashlib, random, shutil, collections, contextlib, io
import numpy as np, cv2, yaml
from PIL import Image

IN_COLAB = "google.colab" in sys.modules
DATASET_SLUG    = "umuttuygurr/aerial-uav-thermal-inferred-unified-dataset"
DATASET_VERSION = None                        # ana notebook'taki ile AYNI olmalı
DATA_DIR   = "/content/uav_thermal_dataset"
DRIVE_ROOT = "/content/drive/MyDrive/uav_proje"
SPLIT_DIR  = f"{DRIVE_ROOT}/split_v1"         # ana notebook'un dondurduğu bölme
WEIGHTS    = f"{DRIVE_ROOT}/runs/uav_thermal_6cls/weights/best.pt"
REQUIRE_SPLIT_MATCH = True                    # ağırlığın eğitildiği bölme doğrulanamazsa dur

NAMES = {0: "person", 1: "car", 2: "bicycle", 3: "other_vehicle", 4: "drone", 5: "mine"}
IMGSZ, CONF, IOU, MAX_DET, BATCH = 640, 0.001, 0.7, 300, 16   # Ultralytics val varsayılanları
SMALL_AREA = 32 * 32                          # COCO "small": alan <= 32x32 px (orijinal görüntü pikseli)
OUT_DIR   = "/content/test_results"
DRIVE_OUT = f"{DRIVE_ROOT}/test_results"
N_SAMPLES, SAMPLE_CONF, SEED = 20, 0.25, 42   # örnek çıktı görselleri

if IN_COLAB and not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(OUT_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
# ==== VERİ + DONDURULMUŞ BÖLME + AĞIRLIK/BÖLME UYUMU ====
if not os.path.isdir(DATA_DIR) or not os.listdir(DATA_DIR):
    import kagglehub
    src = kagglehub.dataset_download(DATASET_SLUG + (f"/versions/{DATASET_VERSION}" if DATASET_VERSION else ""))
    shutil.copytree(src, DATA_DIR, dirs_exist_ok=True)

def fingerprint(rows):
    h = hashlib.sha256()
    for r in sorted(rows, key=lambda r: r["image"]):
        h.update(f'{r["image"]},{r["md5"]},{r["split"]}\n'.encode())
    return h.hexdigest()[:16]

with open(os.path.join(SPLIT_DIR, "manifest.csv"), newline="") as fh:
    rows = list(csv.DictReader(fh))
FP = fingerprint(rows)
info = json.load(open(os.path.join(SPLIT_DIR, "split_info.json")))
assert info["fingerprint"] == FP, "manifest.csv ile split_info.json uyuşmuyor"

# Ağırlık bu bölmeyle mi eğitildi? (train_dir/split_info.json eğitim hücresi tarafından yazılır)
run_info = os.path.join(os.path.dirname(os.path.dirname(WEIGHTS)), "split_info.json")
if os.path.exists(run_info):
    run_fp = json.load(open(run_info)).get("fingerprint")
    if run_fp != FP:
        raise RuntimeError(f"best.pt {run_fp} bölmesiyle eğitilmiş, mevcut bölme {FP}. Bu test kümesi o model için "
                           f"sızıntılı olabilir -> modeli bu bölmeyle yeniden eğit.")
    print(f"✓ Ağırlık bu bölmeyle eğitilmiş (parmak izi {FP})")
else:
    msg = (f"{run_info} yok: best.pt'nin hangi bölmeyle eğitildiği doğrulanamıyor. Eski bölmeyle eğitilmiş bir "
           f"model yeni test kümesindeki görselleri eğitimde görmüş olabilir.")
    if REQUIRE_SPLIT_MATCH:
        raise RuntimeError(msg + " Ana notebook'taki eğitim hücresiyle yeniden eğit ya da bilerek REQUIRE_SPLIT_MATCH=False yap.")
    print("UYARI:", msg)

# Test görsellerinin içeriği manifestle birebir aynı mı?
def md5_file(p):
    with open(p, "rb") as fh:
        return hashlib.md5(fh.read()).hexdigest()
test_rows = sorted((r for r in rows if r["split"] == "test"), key=lambda r: r["image"])
bad = [r["image"] for r in test_rows
       if not os.path.exists(os.path.join(DATA_DIR, r["image"])) or md5_file(os.path.join(DATA_DIR, r["image"])) != r["md5"]]
assert not bad, f"{len(bad)} test görseli eksik/değişmiş (örn. {bad[:3]}) -> DATASET_VERSION'ı kontrol et"

LIST_DIR = os.path.join(OUT_DIR, "lists")
os.makedirs(LIST_DIR, exist_ok=True)
for s in ("train", "val", "test"):          # Ultralytics yaml'da train/val anahtarını da ister; sadece test kullanılır
    paths = sorted(os.path.join(DATA_DIR, r["image"]) for r in rows if r["split"] == s)
    with open(os.path.join(LIST_DIR, f"{s}.txt"), "w") as fh:
        fh.write("\n".join(paths) + "\n")
DATA_YAML = os.path.join(OUT_DIR, "data_test.yaml")
with open(DATA_YAML, "w") as fh:
    yaml.safe_dump({"train": f"{LIST_DIR}/train.txt", "val": f"{LIST_DIR}/val.txt",
                    "test": f"{LIST_DIR}/test.txt", "names": NAMES}, fh, sort_keys=False)
TEST_IMGS = [os.path.join(DATA_DIR, r["image"]) for r in test_rows]
TEST_LBLS = [os.path.join(DATA_DIR, r["label"]) for r in test_rows]
print(f"Test görseli: {len(TEST_IMGS)} | bölme parmak izi: {FP}")

Using Colab cache for faster access to the 'aerial-uav-thermal-inferred-unified-dataset' dataset.
✓ Ağırlık bu bölmeyle eğitilmiş (parmak izi da43dd521edebb0d)
Test görseli: 2974 | bölme parmak izi: da43dd521edebb0d


In [ ]:
# ==== DEĞERLENDİRME: mAP50, mAP50-95, sınıf bazında mAP50, küçük nesne AP -> metrics_test.csv ====
import gc
import torch
from ultralytics import YOLO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# Önceki çalıştırmalardan kalan belleği temizle
gc.collect()
torch.cuda.empty_cache()

model = YOLO(WEIGHTS)
assert {int(k): v for k, v in model.names.items()} == NAMES, f"Model sınıfları farklı: {model.names}"

# --- (1) Genel + sınıf bazında metrikler: Ultralytics val, split="test" ---
res = model.val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=BATCH, conf=CONF, iou=IOU, max_det=MAX_DET,
                plots=True, project=OUT_DIR, name="ultralytics_val", exist_ok=True)
box = res.box
U_MAP50, U_MAP = float(box.map50), float(box.map)
U_CLS = {int(c): (float(box.ap50[i]), float(box.ap[i])) for i, c in enumerate(box.ap_class_index)}

# model.val() sonrası VRAM'i boşalt
del res, box
gc.collect()
torch.cuda.empty_cache()

# --- (2) Küçük nesne AP: COCO alan filtresi (pycocotools) ---
def yolo_to_xywh(vals, w, h):
    if len(vals) == 4:                                   # xc yc w h (normalize)
        xc, yc, bw, bh = vals
        return (xc - bw / 2) * w, (yc - bh / 2) * h, bw * w, bh * h
    xs, ys = np.array(vals[0::2]) * w, np.array(vals[1::2]) * h   # poligon etiketi -> kutu
    return xs.min(), ys.min(), xs.max() - xs.min(), ys.max() - ys.min()

img_id = {os.path.abspath(p): i for i, p in enumerate(TEST_IMGS, start=1)}
gt_images, gt_anns = [], []
gt_by_img = collections.defaultdict(list)
for i, (ip, lp) in enumerate(zip(TEST_IMGS, TEST_LBLS), start=1):
    w, h = Image.open(ip).size
    gt_images.append({"id": i, "file_name": os.path.relpath(ip, DATA_DIR), "width": w, "height": h})
    if not os.path.exists(lp):
        continue
    for line in open(lp):
        s = line.split()
        if not s:
            continue
        c = int(float(s[0]))
        if c not in NAMES:
            raise ValueError(f"Test etiketinde tanımsız sınıf {c}: {lp}")
        x, y, bw, bh = map(float, yolo_to_xywh(list(map(float, s[1:])), w, h))
        a = {"id": len(gt_anns) + 1, "image_id": i, "category_id": c + 1,       # COCO id'leri 1'den başlar
             "bbox": [x, y, bw, bh], "area": bw * bh, "iscrowd": 0}
        gt_anns.append(a); gt_by_img[i].append(a)

dets, preds_by_img = [], collections.defaultdict(list)
SAFE_BATCH = min(BATCH, 8)
CHUNK_SIZE = 32

for start in range(0, len(TEST_IMGS), CHUNK_SIZE):
    chunk = TEST_IMGS[start:start + CHUNK_SIZE]
    for r in model.predict(source=chunk, stream=True, imgsz=IMGSZ, conf=CONF, iou=IOU,
                           max_det=MAX_DET, batch=SAFE_BATCH, half=True, verbose=False):
        iid = img_id[os.path.abspath(r.path)]
        b = r.boxes
        if b is None or len(b) == 0:
            continue
        for (x1, y1, x2, y2), sc, c in zip(b.xyxy.cpu().numpy().tolist(), b.conf.cpu().numpy().tolist(),
                                           b.cls.cpu().numpy().astype(int).tolist()):
            d = {"image_id": iid, "category_id": c + 1, "bbox": [x1, y1, x2 - x1, y2 - y1], "score": sc}
            dets.append(d); preds_by_img[iid].append(d)
    torch.cuda.empty_cache()

cats = [{"id": c + 1, "name": n} for c, n in NAMES.items()]
json.dump({"images": gt_images, "annotations": gt_anns, "categories": cats}, open(f"{OUT_DIR}/coco_gt_test.json", "w"))
json.dump(dets, open(f"{OUT_DIR}/predictions_test.json", "w"))

with contextlib.redirect_stdout(io.StringIO()):
    coco_gt = COCO(f"{OUT_DIR}/coco_gt_test.json")
    if dets:
        coco_dt = coco_gt.loadRes(f"{OUT_DIR}/predictions_test.json")
    else:
        coco_dt = COCO(); coco_dt.dataset = {"images": gt_images, "annotations": [], "categories": cats}; coco_dt.createIndex()
    E = COCOeval(coco_gt, coco_dt, "bbox")
    E.params.maxDets = [1, 10, MAX_DET]                     # Ultralytics ile aynı (300)
    E.params.areaRng = [[0, 1e10], [0, SMALL_AREA], [SMALL_AREA, 96 ** 2], [96 ** 2, 1e10]]
    E.evaluate(); E.accumulate()

def coco_ap(iou=None, area="all", cat=None):
    p = E.eval["precision"][:, :, :, E.params.areaRngLbl.index(area), -1]   # [IoU, recall, sınıf]
    if iou is not None:
        p = p[[int(np.argmin(np.abs(E.params.iouThrs - iou)))]]
    if cat is not None:
        p = p[:, :, [E.params.catIds.index(cat + 1)]]
    p = p[p > -1]                                          # -1: bu alanda GT yok
    return float(p.mean()) if p.size else float("nan")

n_gt = collections.Counter(a["category_id"] - 1 for a in gt_anns)
n_small = collections.Counter(a["category_id"] - 1 for a in gt_anns if a["area"] <= SMALL_AREA)

# --- (3) metrics_test.csv ---
out_rows = []
def add(metric, cid, value, source, note=""):
    nan = value is None or (isinstance(value, float) and np.isnan(value))
    out_rows.append({"metric": metric, "class_id": "all" if cid is None else cid,
                     "class_name": "all" if cid is None else NAMES[cid],
                     "value": "" if nan else (round(float(value), 4) if isinstance(value, float) else value),
                     "source": source, "note": note or ("testte bu sınıfın GT'si yok" if nan else "")})

add("mAP50", None, U_MAP50, "ultralytics")
add("mAP50-95", None, U_MAP, "ultralytics")
add("AP_small_50-95", None, coco_ap(area="small"), "pycocotools", f"COCO APs, alan<={SMALL_AREA}px, maxDets={MAX_DET}")
add("AP_small_50", None, coco_ap(iou=0.5, area="small"), "pycocotools", f"alan<={SMALL_AREA}px, IoU=0.50")
for c in NAMES:
    add("mAP50", c, U_CLS.get(c, (float("nan"),))[0], "ultralytics")
for c in NAMES:
    add("mAP50-95", c, U_CLS.get(c, (None, float("nan")))[1], "ultralytics")
for c in NAMES:
    add("AP_small_50-95", c, coco_ap(area="small", cat=c), "pycocotools",
        "" if n_small[c] else "testte bu sınıfın küçük nesnesi yok")
    add("AP_small_50", c, coco_ap(iou=0.5, area="small", cat=c), "pycocotools",
        "" if n_small[c] else "testte bu sınıfın küçük nesnesi yok")
add("COCO_mAP50", None, coco_ap(iou=0.5), "pycocotools", "çapraz kontrol (101 nokta enterpolasyon)")
add("COCO_mAP50-95", None, coco_ap(), "pycocotools", "çapraz kontrol")
add("n_images", None, len(TEST_IMGS), "count")
add("n_gt", None, len(gt_anns), "count")
add("n_gt_small", None, sum(n_small.values()), "count", f"alan<={SMALL_AREA}px")
for c in NAMES:
    add("n_gt", c, n_gt[c], "count"); add("n_gt_small", c, n_small[c], "count")
add("split_fingerprint", None, FP, "manifest")
add("weights", None, WEIGHTS, "config")

CSV_PATH = os.path.join(OUT_DIR, "metrics_test.csv")
with open(CSV_PATH, "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=["metric", "class_id", "class_name", "value", "source", "note"])
    w.writeheader(); w.writerows(out_rows)

print(f"\n=== TEST ({len(TEST_IMGS)} görsel, {len(gt_anns)} nesne, {sum(n_small.values())} küçük) ===")
print(f"mAP50          : {U_MAP50:.4f}")
print(f"mAP50-95       : {U_MAP:.4f}")
print(f"AP_small 50-95 : {coco_ap(area='small'):.4f}   (COCO, alan <= 32x32 px)")
print(f"AP_small 50    : {coco_ap(iou=0.5, area='small'):.4f}")
print(f"{'sınıf':15s} {'mAP50':>7s} {'AP_s50-95':>10s} {'n_gt':>6s} {'n_small':>8s}")
for c, n in NAMES.items():
    a50 = U_CLS.get(c, (float("nan"),))[0]; aps = coco_ap(area="small", cat=c)
    print(f"{n:15s} {a50:7.4f} {aps:10.4f} {n_gt[c]:6d} {n_small[c]:8d}")
print("\nYazıldı:", CSV_PATH)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.212 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
Model summary (fused): 73 layers, 3,133,922 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3118.7±877.3 MB/s, size: 567.5 KB)
val: Scanning /content/uav_thermal_dataset/merged_uav_dataset/labels/test... 2974 images, 97 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2974/2974 994.7it/s 3.0s
val: New cache created: /content/uav_thermal_dataset/merged_uav_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 186/186 10.8it/s 17.2s
                   all       2974       9611      0.935      0.789      0.897

In [ ]:
import os
import csv
import shutil
import random
from collections import Counter

# ==== 4. DOĞAL DİL RAPORLAMA MODÜLÜ (report_generator.py oluşturma ve içe aktarma) ====
report_module_code = '''from collections import Counter

def generate_report(detections: list) -> str:
    if not detections:
        return "Gözlem sahasında herhangi bir nesne tespit edilmemiştir."
    counts = Counter(detections)
    parts = [f"{count} adet {cls_name}" for cls_name, count in counts.items()]
    summary = parts[0] if len(parts) == 1 else ", ".join(parts[:-1]) + " ve " + parts[-1]
    return f"Gözlem sahasında {summary} tespit edilmiştir."
'''
with open("report_generator.py", "w", encoding="utf-8") as f:
    f.write(report_module_code)

from report_generator import generate_report

# ==== 20 ÖRNEK ÇIKTI (deterministik seçim; yeşil = GT, kırmızı = tahmin) + DRIVE'A KOPYALA ====
SAMPLE_DIR = os.path.join(OUT_DIR, "ornek_ciktilar")
shutil.rmtree(SAMPLE_DIR, ignore_errors=True); os.makedirs(SAMPLE_DIR)
rng = random.Random(SEED)
pool = sorted(gt_by_img)                                                    # GT içeren test görselleri
small_pool = [i for i in pool if any(a["area"] <= SMALL_AREA for a in gt_by_img[i])]
pick = rng.sample(small_pool, min(N_SAMPLES // 2, len(small_pool)))         # yarısı küçük nesneli
rest = [i for i in pool if i not in set(pick)]
pick = sorted(pick + rng.sample(rest, min(N_SAMPLES - len(pick), len(rest))))

index = []
rapor_satirlari = []

for n, iid in enumerate(pick, start=1):
    im = cv2.imread(TEST_IMGS[iid - 1])
    for a in gt_by_img[iid]:
        x, y, w, h = map(round, a["bbox"])
        cv2.rectangle(im, (x, y), (x + w, y + h), (0, 255, 0), 1)

    shown = [d for d in preds_by_img[iid] if d["score"] >= SAMPLE_CONF]
    tespit_siniflari = []

    for d in shown:
        x, y, w, h = map(round, d["bbox"])
        cls_name = NAMES[d["category_id"] - 1]
        tespit_siniflari.append(cls_name)

        cv2.rectangle(im, (x, y), (x + w, y + h), (0, 0, 255), 1)
        cv2.putText(im, f'{cls_name} {d["score"]:.2f}', (x, max(y - 3, 9)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 255), 1, cv2.LINE_AA)

    durum_raporu = generate_report(tespit_siniflari)

    cv2.putText(im, "yesil=GT  kirmizi=tahmin", (4, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)
    fname = f"{n:02d}_{os.path.splitext(os.path.basename(TEST_IMGS[iid - 1]))[0]}.jpg"
    cv2.imwrite(os.path.join(SAMPLE_DIR, fname), im)

    print(f"[{fname}] -> {durum_raporu}")
    rapor_satirlari.append(f"{fname}: {durum_raporu}")

    index.append({
        "file": fname,
        "image": gt_images[iid - 1]["file_name"],
        "n_gt": len(gt_by_img[iid]),
        "n_gt_small": sum(a["area"] <= SMALL_AREA for a in gt_by_img[iid]),
        f"n_pred@{SAMPLE_CONF}": len(shown),
        "rapor": durum_raporu
    })

with open(os.path.join(SAMPLE_DIR, "index.csv"), "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=list(index[0])); w.writeheader(); w.writerows(index)

with open(os.path.join(SAMPLE_DIR, "raporlar.txt"), "w", encoding="utf-8") as rf:
    rf.write("\n".join(rapor_satirlari))

print(f"{len(pick)} örnek kaydedildi: {SAMPLE_DIR}")

os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copy2("report_generator.py", DRIVE_OUT)
for f in ("metrics_test.csv", "coco_gt_test.json", "predictions_test.json"):
    shutil.copy2(os.path.join(OUT_DIR, f), DRIVE_OUT)
for d in ("ornek_ciktilar", "ultralytics_val"):
    if os.path.isdir(os.path.join(OUT_DIR, d)):
        shutil.copytree(os.path.join(OUT_DIR, d), os.path.join(DRIVE_OUT, d), dirs_exist_ok=True)
print("Drive'a kopyalandı:", DRIVE_OUT)

[01_02_hituav__1_60_30_0_00090__3f55603120.jpg] -> Gözlem sahasında 6 adet person tespit edilmiştir.
[02_03_llvip__120179__b620d3d0ed.jpg] -> Gözlem sahasında 4 adet car ve 1 adet person tespit edilmiştir.
[03_03_llvip__120274__b933da4a1f.jpg] -> Gözlem sahasında 4 adet car ve 2 adet person tespit edilmiştir.
[04_03_llvip__120326__c7a24636dd.jpg] -> Gözlem sahasında 3 adet car ve 2 adet person tespit edilmiştir.
[05_01_uav2uav__01724__218eff6b90.jpg] -> Gözlem sahasında 1 adet drone tespit edilmiştir.
[06_01_uav2uav__02006__50ed4e02dd.jpg] -> Gözlem sahasında 1 adet drone tespit edilmiştir.
[07_01_uav2uav__02202__c5c7f0f51b.jpg] -> Gözlem sahasında 3 adet drone tespit edilmiştir.
[08_01_uav2uav__02243__badce0a7db.jpg] -> Gözlem sahasında 2 adet drone tespit edilmiştir.
[09_01_uav2uav__02244__b8378f0b59.jpg] -> Gözlem sahasında 2 adet drone tespit edilmiştir.
[10_01_uav2uav__974__8abc3be555.jpg] -> Gözlem sahasında 2 adet drone tespit edilmiştir.
[11_02_hituav__0_60_40_0_01660__9ec44a4a